# 🎓 Reinforcement Learning Lecture 2: Exercises & Detailed Solutions

**Course:** Data & A.I. 6 (DAI6)  
**Instructor:** Alexander Michielsen  
**Study Guide & Solutions:** Step-by-Step Explanations with Code, Math & Intuitive Analogies

---

## 🧭 Theoretical Foundations & Quick Refresh (From RL1 & RL2 Notes)

Before diving into the exercises, let's establish the fundamental concepts in simple terms:

### 1. What is Reinforcement Learning (RL)?
In standard Machine Learning (Supervised Learning), a model is given labeled answers (e.g., "this picture is a cat").  
In **Reinforcement Learning**, there are **no answers and no supervisor**. The agent learns like a human toddler learning to walk:
1. **Agent (عامل)** takes an **Action (عمل)** ($a_t$).
2. **Environment (محیط)** reacts: shifts to a new **State (وضعیت)** ($s_{t+1}$) and emits a scalar **Reward (پاداش)** ($r_{t+1}$).
3. **Goal:** Maximize total cumulative discounted reward ($G_t = \sum \gamma^k r_{t+k+1}$), not just immediate gratification (**The Sutton Reward Hypothesis**).

### 2. State-Value $V(s)$ vs. Action-Value $Q(s, a)$
- **State-Value $V(s)$:** "How good is it to be in this room/state?"
- **Action-Value $Q(s, a)$:** "If I am in this room, how good is it to take the *door on the right*?"  
  Because an agent must decide *which action to take* without knowing the environment's internal laws (Model-Free RL), having $Q(s, a)$ is much more practical!

### 3. Temporal Difference (TD) vs. Monte Carlo (MC)
- **Monte Carlo (MC):** Waits until the entire game/episode ends, calculates the total score $G_t$, and updates values. (Like a student who only gets feedback after the final semester exam).
- **Temporal Difference (TD) / Q-Learning:** Updates after **every single step**! It uses **bootstrapping (خودراه‌اندازی)** (updating a guess based on another guess):
  $$\text{New } Q(s, a) = \text{Old } Q(s, a) + \alpha \times \big[ \underbrace{r + \gamma \max_{a'} Q(s', a')}_{\text{TD Target}} - \text{Old } Q(s, a) \big]$$


In [1]:
# Set directory to reinforcement starter package
import os
import sys

# Ensure we are working inside the reinforcement directory
if os.path.exists("reinforcement"):
    os.chdir("reinforcement")
print("Active Working Directory:", os.getcwd())


Active Working Directory: /Users/nooshin/Uni/Data/data6/data6/ReinforcementLearning/week1/reinforcement


---

# 📝 Exercise 1: Q-Learning Algorithm & Code Mapping

### The Theoretical Algorithm (from Sutton & Barto):
```text
Q-learning (off-policy TD control) for estimating pi ~ pi_*

Algorithm parameters: step size alpha in (0, 1], small epsilon > 0
Initialize Q(s, a), for all s in S+, a in A(s), arbitrarily except that Q(terminal, .) = 0
Loop for each episode:
    Initialize S
    Loop for each step of episode:
        Choose A from S using policy derived from Q (e.g., epsilon-greedy)
        Take action A, observe R, S'
        Q(S, A) <- Q(S, A) + alpha * [R + gamma * max_a Q(S', a) - Q(S, A)]
        S <- S'
    until S is terminal
```

---

### 🔍 Question 1: Finding Algorithm Elements in `qlearningAgents.py`

Let's examine exactly how the theory maps directly to the provided code in `qlearningAgents.py`:

| Mathematical Element in Algorithm | Meaning in Plain English | Corresponding Implementation in `qlearningAgents.py` |
| :--- | :--- | :--- |
| **Parameters: $\alpha, \varepsilon, \gamma$** | Learning rate $\alpha$, Exploration rate $\varepsilon$, Discount factor $\gamma$ | Initialized in constructor: `self.alpha`, `self.epsilon`, `self.discount` |
| **Initialize $Q(s, a)$** | A table storing estimated values for every state-action pair, defaulting to $0.0$ | Line 44: `self.Q = util.Counter()`<br/>Lines 54-58: `getQValue(state, action)` returns `0.0` if unseen |
| **Choose $A$ using $\varepsilon$-greedy** | With probability $\varepsilon$ explore randomly, else choose greedy best $\max Q$ | Lines 90-104: `getAction(state)`<br/>Uses `util.flipCoin(self.epsilon)` and `computeActionFromQValues(state)` |
| **Take action $A$, observe $R, S'$** | Step the simulator forward, get reward $R$ and next state $S'$ | Handled by simulation engine; delivered to agent via `observeTransition` and `update(state, action, nextState, reward)` |
| **$\max_a Q(S', a)$** | The highest possible Q-value among legal actions in the next state | Lines 61-69: `computeValueFromQValues(nextState)` |
| **TD Update Rule:**<br/>$Q \leftarrow Q + \alpha [R + \gamma \max Q' - Q]$ | Adjust current estimate by fraction $\alpha$ of the prediction error (TD Error) | Lines 115-116 in `update()`:<br/>`current_q = self.getQValue(state, action)`<br/>`self.Q[state][action] = current_q + self.alpha * (reward + self.discount * self.computeValueFromQValues(nextState) - current_q)` |
| **Loop until terminal** | Stop updating when state is terminal | Lines 68-69: returns `0.0` if no legal actions remain |


In [ ]:
# Let's inspect the exact implementation of update and getAction in qlearningAgents.py
import inspect
import qlearningAgents

print("--- Code of QLearningAgent.update ---")
print(inspect.getsource(qlearningAgents.QLearningAgent.update))

print("--- Code of QLearningAgent.getAction ---")
print(inspect.getsource(qlearningAgents.QLearningAgent.getAction))


---

### 🧮 Mathematical Proof: Why do the Q-values become 0.05, 0.25, 0.62, 0.94?

In Exercise 1, the document states:
> *"If you manually steer the blue ball north and then east along the optimal path for four episodes without noise, you should see the following Q-values:*  
> *(0,2) East: **0.05**, (1,2) East: **0.25**, (2,2) East: **0.62**, (3,2) Exit: **0.94**"*

#### Why does this happen? (The Concept of Backward Information Propagation)
In Q-Learning, the agent receives **zero living reward** ($R = 0$) while moving through the grid, until it reaches the goal at `(3,2)` where $R = +1.0$.
- Hyperparameters in `gridworld.py`: $\alpha = 0.5$, $\gamma = 0.9$.
- Update formula: $Q(s, a) \leftarrow Q(s, a) + 0.5 \times \big[ R + 0.9 \times \max_{a'} Q(s', a') - Q(s, a) \big]$.

Let's trace the 4 episodes step-by-step:

#### **Episode 1:**
- All states start at $Q = 0.0$.
- Moves `(0,0) -> (0,1) -> (0,2) -> (1,2) -> (2,2)` all have $R = 0$ and next state value $\max Q = 0$, so their Q-values remain **$0.0$**.
- From `(2,2)` moving East enters `(3,2)`. Because $Q((3,2), \text{exit})$ is still $0.0$, $Q((2,2), \text{East})$ is still **$0.0$**.
- Finally, from `(3,2)` the agent takes `exit`, receiving $R = +1.0$ into terminal state ($V = 0$):
  $$Q((3,2), \text{exit}) \leftarrow 0.0 + 0.5 \times [1.0 + 0 - 0.0] = \mathbf{0.50}$$

#### **Episode 2:**
- Now `(3,2)` has value $0.50$!
- When the agent moves from `(2,2)` East into `(3,2)`:
  $$Q((2,2), \text{East}) \leftarrow 0.0 + 0.5 \times [0 + 0.9 \times 0.50 - 0.0] = 0.5 \times 0.45 = \mathbf{0.225}$$
- In `(3,2)`, exit receives $R = 1.0$:
  $$Q((3,2), \text{exit}) \leftarrow 0.50 + 0.5 \times [1.0 - 0.50] = \mathbf{0.75}$$

#### **Episode 3:**
- Now `(2,2)` East has value $0.225$!
- When moving from `(1,2)` East into `(2,2)`:
  $$Q((1,2), \text{East}) \leftarrow 0.0 + 0.5 \times [0 + 0.9 \times 0.225 - 0.0] = \mathbf{0.1013}$$
- When moving from `(2,2)` East into `(3,2)`:
  $$Q((2,2), \text{East}) \leftarrow 0.225 + 0.5 \times [0 + 0.9 \times 0.75 - 0.225] = \mathbf{0.45}$$
- In `(3,2)`, exit updates:
  $$Q((3,2), \text{exit}) \leftarrow 0.75 + 0.5 \times [1.0 - 0.75] = \mathbf{0.875}$$

#### **Episode 4:**
- Now `(1,2)` East has value $0.1013$!
- When moving from `(0,2)` East into `(1,2)`:
  $$Q((0,2), \text{East}) \leftarrow 0.0 + 0.5 \times [0 + 0.9 \times 0.1013 - 0.0] = \mathbf{0.0456} \approx \mathbf{0.05}$$
- When moving from `(1,2)` East into `(2,2)`:
  $$Q((1,2), \text{East}) \leftarrow 0.1013 + 0.5 \times [0 + 0.9 \times 0.45 - 0.1013] = \mathbf{0.2531} \approx \mathbf{0.25}$$
- When moving from `(2,2)` East into `(3,2)`:
  $$Q((2,2), \text{East}) \leftarrow 0.45 + 0.5 \times [0 + 0.9 \times 0.875 - 0.45] = \mathbf{0.6188} \approx \mathbf{0.62}$$
- In `(3,2)`, exit updates:
  $$Q((3,2), \text{exit}) \leftarrow 0.875 + 0.5 \times [1.0 - 0.875] = \mathbf{0.9375} \approx \mathbf{0.94}$$

🎯 **Key Intuition:** Knowledge of the terminal reward travels backward by exactly **1 step per episode**! This is why TD learning requires multiple episodes for value to propagate across the entire grid.


In [ ]:
# Let's run a simulation in Python verifying this exact mathematical derivation!
alpha = 0.5
gamma = 0.9
q = {}

def get_q(s, a):
    return q.get((s, a), 0.0)

def get_v(s):
    if s == 'TERM': return 0.0
    actions = ['exit'] if s == (3,2) else ['north', 'east', 'south', 'west']
    return max([get_q(s, a) for a in actions] + [0.0])

def update(s, a, s_next, r):
    cur = get_q(s, a)
    target = r + gamma * get_v(s_next)
    q[(s, a)] = cur + alpha * (target - cur)

path = [
    ((0,0), 'north', (0,1), 0.0),
    ((0,1), 'north', (0,2), 0.0),
    ((0,2), 'east', (1,2), 0.0),
    ((1,2), 'east', (2,2), 0.0),
    ((2,2), 'east', (3,2), 0.0),
    ((3,2), 'exit', 'TERM', 1.0)
]

print(f"{'Episode':<9} | {'(0,2) East':<12} | {'(1,2) East':<12} | {'(2,2) East':<12} | {'(3,2) Goal':<12}")
print("-" * 65)

for ep in range(1, 5):
    for s, a, s_next, r in path:
        update(s, a, s_next, r)
    print(f"{ep:<9} | {get_q((0,2), 'east'):<12.4f} | {get_q((1,2), 'east'):<12.4f} | {get_q((2,2), 'east'):<12.4f} | {get_q((3,2), 'exit'):<12.4f}")


### 🕹️ Experimenting with Gridworld Options

Let's answer the second part of Exercise 1:
1. **Removing `-m` (Manual mode):**
   - In manual mode (`-m`), you drove the agent with the keyboard.
   - When `-m` is removed, the agent relies on its own $\varepsilon$-greedy policy (`self.getAction(state)`). Initially, with $Q=0$, it wanders randomly. Once it stumbles upon the $+1$ goal, positive Q-values begin backpropagating and guiding it.
2. **Increasing episodes (`-k`):**
   - With $k=5$, only states near the goal receive meaningful values.
   - With $k=100$, Q-values converge to their true theoretical values ($V(s) = \gamma^{\text{distance}} \times 1.0$), and the optimal path is clearly visible from every cell.
3. **Changing `--noise` (e.g., `--noise 0.2`):**
   - With `noise 0.0`, moves are 100% deterministic.
   - With `noise 0.2`, 80% of the time the agent moves in the intended direction, and 10% each it slips sideways (just like `is_slippery=True` in FrozenLake!).
   - **Behavioral effect:** Notice how the Q-value of moving East in `(3,1)` (right next to the $-1$ hazard pit) drops significantly because the risk of accidentally slipping into the hazard is factored into the Q-value!


In [ ]:
# Run manual control in Gridworld (5 episodes, no noise)
!python gridworld.py -a q -k 5 -m --noise 0.0


In [ ]:
# Run autonomous Gridworld with 50 episodes and noise = 0.2
!python gridworld.py -a q -k 50 --noise 0.2


---

# 🤖 Exercise 2: Epsilon-Greedy & The Crawler Robot

### 1. Where is $\varepsilon$-greedy implemented in the code?
In `qlearningAgents.py`, inside the `getAction(self, state)` method:

```python
def getAction(self, state):
    legalActions = self.getLegalActions(state)
    action = None
    if legalActions:
        # If flipCoin(epsilon) is True: EXPLORE (pick random action)
        # If flipCoin(epsilon) is False: EXPLOIT (pick best action from Q-table)
        action = self.computeActionFromQValues(state) if not util.flipCoin(self.epsilon) else random.choice(legalActions)
    return action
```

#### 🍕 The Intuitive Analogy: "The Restaurant Dilemma"
- **Exploitation ($1 - \varepsilon$):** Going to your favorite restaurant where you know the pizza is an 8/10. Safe, reliable, good reward.
- **Exploration ($\varepsilon$):** Trying a new sushi place down the street. It might be terrible (1/10), but it might be legendary (10/10)!
- Without exploration, you will never discover the 10/10 restaurant. Without exploitation, you eat bad food half the time!

---

### 2. The Crawler Robot (`crawler.py`)

Run the crawler below:
```bash
python crawler.py
```

#### What is happening inside the Crawler?
The crawler is a simulated physical robot with two articulated joints:
1. **Arm angle**
2. **Hand angle**

- **State Space:** Continuous joint angles are discretized into grid cells (e.g., $9 \times 9 = 81$ discrete joint configurations).
- **Action Space:** Move arm up/down, move hand up/down.
- **Reward Signal:** Distance moved forward horizontally to the right!
- **How it learns:** At first, the robot twitches randomly (motor babbling). Gradually, Q-learning discovers that a synchronized **wave-like crawling motion** (push down, extend, lift, retract) moves the body forward!

#### Role of the Parameters:
- **Step Delay:** Controls the animation speed in the GUI. A low step delay lets it learn 100x faster; a high delay lets humans watch the motion.
- **Learning Rate $\alpha$:** How rapidly new experiences update the Q-table.
  - High $\alpha$ (0.8): Robot quickly adapts to new positions, but values can oscillate.
  - Low $\alpha$ (0.1): Smooth, stable, but slower learning.
- **Epsilon $\varepsilon$:** Exploration rate.
  - $\varepsilon = 1.0$: Robot flails around completely randomly.
  - $\varepsilon = 0.0$: Robot strictly follows its best known sequence. Once learned, setting $\varepsilon \to 0$ makes it crawl forward smoothly without erratic twitches.
- **Discount Factor $\gamma$:** Patience / Horizon.
  - Crawling requires a **cycle of 4 sequential steps** (prepare arm, plant hand, pull body, reset).
  - If $\gamma = 0$ (impatient), the robot only cares about the immediate millimeter of movement and will never discover the multi-step crawling cycle!
  - A high $\gamma$ (0.9–0.99) is essential to link the multi-step cycle together.


In [ ]:
# Run the crawler robot GUI
# Adjust learning rate, epsilon, and step delay using the sliders in the GUI window!
!python crawler.py


---

# 👻 Exercise 3: Q-Learning and Pacman (Exploration vs. Exploitation & The Scalability Wall)

In this exercise, we train Pacman with tabular Q-learning on two layouts:
1. `smallGrid`
2. `mediumGrid`

```bash
python pacman.py -p PacmanQAgent -x 2000 -n 2010 -l smallGrid
```
- `-x 2000`: 2,000 **training games** (runs in quiet mode without GUI).
- `-n 2010`: 2,010 total games (meaning the last 10 games are **test games** displayed in GUI).

---

### 1. Training Phase vs. Testing Phase (Exploration vs. Exploitation)

In `learningAgents.py`, observe how the agent automatically switches behavior:
```python
def stopEpisode(self):
    self.episodesSoFar += 1
    if self.episodesSoFar >= self.numTraining:
        # Take off the training wheels!
        self.epsilon = 0.0    # No exploration (pure exploitation)
        self.alpha = 0.0      # No learning (frozen weights)
```

- **During Training ($x \le 2000$):**
  - $\varepsilon > 0$: Pacman explores randomly. Because he makes random moves, he frequently walks straight into ghosts! That is why average rewards are negative for the first 1,000 games.
- **During Testing ($x > 2000$):**
  - $\varepsilon = 0.0$: Exploration is completely turned off. Pacman acts purely greedily ($\pi^*(s) = \arg\max_a Q(s, a)$).
  - On `smallGrid`, he wins $\ge 90\%$ of the test games!

---

### 2. The Scalability Wall: Why does it fail miserably on `mediumGrid`?

When you swap `smallGrid` with `mediumGrid`, Pacman fails completely. Even after thousands of games, he still plays poorly and loses.

#### 💥 The Culprit: The Curse of Dimensionality (نفرین ابعاد)

In Tabular Q-learning, the Q-table has one row for **every unique state**:
$$\text{State} = \big( \text{Pacman position } (x, y), \; \text{Ghost positions } (x_g, y_g), \; \text{Grid of remaining food pellets} \big)$$

Let's calculate the size of the state space:
- **On `smallGrid` ($3 \times 4$ with 2 food pellets):**
  - Pacman positions: $\approx 6$
  - Ghost positions: $\approx 6$
  - Food combinations: $2^2 = 4$
  - Total states: $6 \times 6 \times 4 = 144$ states.
  - An agent playing 2,000 games visits every state dozens of times. The table converges easily!

- **On `mediumGrid` ($20 \times 11$ with $\approx 30$ food pellets):**
  - Pacman positions: $\approx 150$
  - Ghost positions: $150 \times 150 = 22,500$
  - Food combinations: $2^{30} \approx 1,073,741,824$ (over 1 billion combinations!)
  - Total states: $> 10^{15}$ states!

#### 🚫 Why Tabular RL Cannot Generalize:
In a lookup table, state `A` and state `B` are completely independent rows:
- If Pacman learns that dying at position `(5, 5)` next to a ghost is terrible when pellet #1 is present...
- If pellet #1 is eaten, that is a **completely different state**! The Q-table has $0.0$ for it and knows **nothing**!
- Pacman would need billions of games to visit every state even once.

💡 **Solution Preview (Lecture 3 & Deep RL):**  
Instead of an exact table, we use **Function Approximation** (linear features or Deep Neural Networks like **DQN**). A neural network learns general concepts like *"distance to nearest ghost"* and generalizes across all boards!


In [ ]:
# Train Pacman on smallGrid (2000 training, 10 testing)
!python pacman.py -p PacmanQAgent -x 2000 -n 2010 -l smallGrid


In [ ]:
# Observe failure of tabular Q-learning on mediumGrid
!python pacman.py -p PacmanQAgent -x 2000 -n 2010 -l mediumGrid


---

# 🎯 Exercise 4: Reward Engineering & Pitfalls

### 1. Where is the reward computed in the code?
In `reinforcement`, inside lines 209-217:

```python
def observationFunction(self, state):
    if not self.lastState is None:
        reward = state.getScore() - self.lastState.getScore()
        self.observeTransition(self.lastState, self.lastAction, state, reward)
    return state
```

The reward is simply the **difference in game score** between two consecutive timesteps:
$$\text{Reward} = \text{Score}(s_{t+1}) - \text{Score}(s_t)$$

#### Breakdown of Default Pacman Game Scoring:
- Taking a step: $-1$ point (forces Pacman to finish quickly).
- Eating a regular food pellet: $+10$ points (net $+9$).
- Eating a ghost: $+200$ points.
- Dying (caught by ghost): $-500$ points.
- Clearing the board (Winning): $+500$ points.

---

### 2. How could we engineer alternative rewards?

While score difference works reasonably well, we can design much better reward signals (**Reward Shaping**):

1. **Food Proximity Reward (Distance Guidance):**
   - Give a small positive reward when Pacman decreases Manhattan distance to the closest food:
     $$R_{\text{food}} = +0.5 \text{ if closer to nearest food, else } -0.5$$
   - *Advantage:* Speeds up early learning dramatically by providing dense feedback instead of waiting to blindly bump into food.

2. **Ghost Danger Field (Safety Buffer):**
   - If distance to an active ghost $\le 2$ tiles, incur a progressive penalty:
     $$R_{\text{danger}} = -\frac{10}{\text{dist}(\text{Pacman}, \text{Ghost})}$$
   - *Advantage:* Teaches Pacman to flee *before* the ghost touches him.

3. **Scared Ghost Hunting Incentive:**
   - When ghosts eat a power capsule and turn blue/scared, reverse the danger penalty into an attraction bonus!

---

### ⚠️ Critical Warning: The Danger of Reward Hacking (The "Hedonistic Immortal")

As covered in **Lecture 2 (Russell & Norvig Gridworld analysis)**:
> *"In Reinforcement Learning, you get what you incentivize, not what you intended!"*

Imagine an AI engineer thinks: *"Pacman dies too fast. Let's give him $+1.0$ for every step he stays alive!"*

**What happens?**
- If living reward is $+1.0$, surviving 1,000 steps gives $+1,000$ points!
- Winning only gives $+500$ points and ends the game (cutting off the reward stream).
- **The Result:** Pacman turns into a **Hedonistic Immortal**! He intentionally refuses to eat the last food pellet, avoids ghosts, and loops around the maze forever to collect infinite points!

Always ensure that your terminal goal delivers the highest cumulative return, and that step costs prevent infinite stalling.


---

## 🧠 Quick Review & Exam Checklist (Key Takeaways)

1. **Q-Learning is Model-Free:** Does not require transition probabilities $\mathcal{P}(s'|s,a)$.
2. **Q-Learning is Off-Policy:** Updates toward the optimal greedy target $\max_{a'} Q(s', a')$ regardless of which exploratory action was taken by $\varepsilon$-greedy.
3. **Bootstrapping:** Updates immediate estimate from downstream estimate ($r + \gamma \max Q'$).
4. **Curse of Dimensionality:** Tabular methods fail on large state spaces because they cannot generalize across unseen states.
5. **Reward Engineering:** Non-terminal penalties (step costs) incentivize swift solutions; living bonuses risk reward hacking.

---
*Created for KdG DAI6 Reinforcement Learning Module.*
